# open-scribe demo

End-to-end walkthrough of the pipeline:

1. **ASR** on a real audio file → transcript with timestamps
2. **SOAP note generation** on a synthetic clinical transcript → structured SoapNote
3. **FHIR Bundle emission** → valid R4B `Patient + Encounter + DocumentReference`

The synthetic clinical transcript in step 2 stands in for a real medical encounter. Once PriMock57 is downloaded, swap it for a real one.

## 1. ASR on a real audio file

Runs faster-whisper on `examples/sample.mp3` (any short WAV/MP3 works). Note: the bundled sample is non-clinical, so downstream SOAP would be empty — that's the *correct* behavior (prompt instructs the LLM not to fabricate clinical content).

In [1]:
from pathlib import Path
from open_scribe.asr import transcribe

# Drop any short clinical audio at ../examples/sample.mp3 to exercise this cell.
# The audio file is NOT committed — this cell will raise FileNotFoundError on a fresh clone.
audio = Path('../examples/sample.mp3')
transcript = transcribe(audio, model_size='tiny.en')

print(f'Language: {transcript.language}  Duration: {transcript.duration:.1f}s  Turns: {len(transcript.turns)}')
print(f'Mean turn length: {sum(t.end - t.start for t in transcript.turns) / max(len(transcript.turns), 1):.2f}s')

[22:45:53] loading whisper model: tiny.en                                                                 ]8;id=820609;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/asr.py\asr.py]8;;\:]8;id=118301;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/asr.py#59\59]8;;\

           transcribing: sample.mp3                                                                       ]8;id=541871;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/asr.py\asr.py]8;;\:]8;id=513039;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/asr.py#62\62]8;;\

Language: en  Duration: 55.3s  Turns: 15
Mean turn length: 3.69s


## 2. SOAP note generation on a synthetic clinical transcript

Set up the LLM backend via env vars before launching Jupyter. Example (Groq):
```bash
export OPEN_SCRIBE_BASE_URL=https://api.groq.com/openai/v1
export OPEN_SCRIBE_API_KEY=$GROQ_API_KEY
export OPEN_SCRIBE_MODEL=llama-3.3-70b-versatile
```

In [2]:
from open_scribe.asr import Transcript, Turn
from open_scribe.note_gen import generate_note

clinical = Transcript(turns=[
    Turn(speaker='clinician', start=0, end=4, text='Good morning. What brings you in today?'),
    Turn(speaker='patient', start=4, end=12, text='Sore throat for three days, and a dry cough. No fever I think, but I feel tired.'),
    Turn(speaker='clinician', start=12, end=15, text='Any trouble swallowing or shortness of breath?'),
    Turn(speaker='patient', start=15, end=20, text='Swallowing hurts. Breathing fine. No chest pain.'),
    Turn(speaker='clinician', start=20, end=25, text='Any sick contacts? Travel?'),
    Turn(speaker='patient', start=25, end=28, text='My son had a cold last week. No travel.'),
    Turn(speaker='clinician', start=28, end=40, text='Temp 38.2, pulse 88. Pharynx mildly erythematous, no exudate. Lungs clear. No cervical lymphadenopathy.'),
    Turn(speaker='clinician', start=40, end=55, text='Looks like a viral URI. No antibiotics needed. Rest, fluids, ibuprofen PRN. Return if high fever, breathing trouble, or not improving in a week.'),
], language='en', duration=55.0)

note = generate_note(clinical)
print(note.model_dump_json(indent=2))

[22:45:58] note_gen: backend=custom model=llama-3.3-70b-versatile                                    ]8;id=978142;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/note_gen.py\note_gen.py]8;;\:]8;id=969994;file:///Users/yashpatil/projects/open-scribe/src/open_scribe/note_gen.py#59\59]8;;\

{
  "subjective": "The patient presents with a sore throat for three days and a dry cough. The patient denies fever but reports feeling tired. Swallowing is painful, but there is no shortness of breath or chest pain. The patient's son had a cold last week, but there has been no travel.",
  "objective": "Temperature is 38.2 degrees, pulse is 88. The pharynx is mildly erythematous with no exudate, lungs are clear, and there is no cervical lymphadenopathy.",
  "assessment": "Viral URI",
  "plan": "The patient is advised to rest, drink fluids, and take ibuprofen as needed. The patient should return if they experience a high fever, breathing trouble, or if their condition does not improve within a week."
}


## 3. FHIR R4B Bundle emission

Wraps the SOAP note in a self-contained transaction Bundle: Patient + Encounter + DocumentReference. The note text is embedded as a base64 attachment on the DocumentReference.

In [3]:
import json
from open_scribe.fhir_emit import to_document_reference

bundle = to_document_reference(note)

print('Bundle type:', bundle['type'])
print('Entries:', [e['resource']['resourceType'] for e in bundle['entry']])
print()
print('DocumentReference.type:')
print(json.dumps(bundle['entry'][2]['resource']['type'], indent=2))

Bundle type: transaction
Entries: ['Patient', 'Encounter', 'DocumentReference']

DocumentReference.type:
{
  "coding": [
    {
      "system": "http://loinc.org",
      "code": "11506-3",
      "display": "Subsequent evaluation note"
    }
  ],
  "text": "Progress note"
}


In [4]:
import base64
encoded = bundle['entry'][2]['resource']['content'][0]['attachment']['data']
print(base64.b64decode(encoded).decode('utf-8'))

SUBJECTIVE
The patient presents with a sore throat for three days and a dry cough. The patient denies fever but reports feeling tired. Swallowing is painful, but there is no shortness of breath or chest pain. The patient's son had a cold last week, but there has been no travel.

OBJECTIVE
Temperature is 38.2 degrees, pulse is 88. The pharynx is mildly erythematous with no exudate, lungs are clear, and there is no cervical lymphadenopathy.

ASSESSMENT
Viral URI

PLAN
The patient is advised to rest, drink fluids, and take ibuprofen as needed. The patient should return if they experience a high fever, breathing trouble, or if their condition does not improve within a week.

